# Notebook 05: Heterogeneous Treatment Effects

**Extension of:** Appel, Pan & Roberts (2023)

This notebook tests whether the party promotion effects documented in the main analysis vary across demographic and attitudinal subgroups. We add **triple interactions** (`Party x Alignment x Moderator`) to the base OLS formula and examine four moderators: **education**, **age**, **gender**, and **political interest**.

### Key equation

$$\text{outcome}_i = \beta_D D_i + \beta_R R_i + \gamma_D (D_i \times \text{ProDem}_i) + \gamma_R (R_i \times \text{ProRep}_i) + \delta_D (D_i \times M_i) + \delta_R (R_i \times M_i) + \lambda_D (D_i \times \text{ProDem}_i \times M_i) + \lambda_R (R_i \times \text{ProRep}_i \times M_i) + \varepsilon_i$$

The triple-interaction terms $\lambda_D$ and $\lambda_R$ are the key quantities of interest. They measure whether the magnitude of party promotion varies with the moderator $M$. Continuous moderators are mean-centered so that the base terms retain their "at the mean" interpretation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from utils import (
    load_main_data, fit_ols_clustered, tidy,
    marginal_effect_at_values, OUTCOME_LABELS
)

pd.set_option('display.max_columns', 20)
%matplotlib inline

OUTCOMES = ['remove', 'harm', 'censorship']

# Load data and mean-center continuous moderators
df = load_main_data()
df['education_c'] = df['education'] - df['education'].mean()
df['age_c'] = df['age'] - df['age'].mean()
df['political_interest_c'] = df['political_interest'] - df['political_interest'].mean()

print(f'Loaded {df.shape[0]} observations, {df["id"].nunique()} respondents')
print(f'Mean education: {df["education"].mean():.2f}, age: {df["age"].mean():.1f}, '
      f'political interest: {df["political_interest"].mean():.2f}')

In [ ]:
# Triple-interaction formula template
HTE_FORMULA = (
    '{outcome} ~ 0 + C(party_id)'
    ' + party_id_dem:headline_pro_dem + party_id_rep:headline_pro_rep'
    ' + party_id_dem:{mod} + party_id_rep:{mod}'
    ' + party_id_dem:headline_pro_dem:{mod} + party_id_rep:headline_pro_rep:{mod}'
)

# Labels for triple-interaction terms
def triple_labels(mod_label):
    return {
        'party_id_dem:headline_pro_dem': f'Dem x ProDem (base)',
        'party_id_rep:headline_pro_rep': f'Rep x ProRep (base)',
        f'party_id_dem:headline_pro_dem:{mod_label}': f'Dem x ProDem x {mod_label}',
        f'party_id_rep:headline_pro_rep:{mod_label}': f'Rep x ProRep x {mod_label}',
    }

def fit_hte_models(mod_var, outcomes=OUTCOMES):
    """Fit HTE models for all three outcomes with a given moderator."""
    results = {}
    for outcome in outcomes:
        formula = HTE_FORMULA.format(outcome=outcome, mod=mod_var)
        res = fit_ols_clustered(formula, df, cluster_var='id', weight_var='weight')
        results[outcome] = res
    return results

def display_triple_terms(results, mod_var):
    """Display a tidy table of triple-interaction terms across outcomes."""
    rows = []
    for outcome, res in results.items():
        td = tidy(res)
        # Keep only triple-interaction terms
        mask = td['term'].str.contains(f':{mod_var}') & td['term'].str.count(':').ge(2)
        sub = td[mask].copy()
        sub.insert(0, 'outcome', OUTCOME_LABELS[outcome])
        rows.append(sub)
    combined = pd.concat(rows, ignore_index=True)
    return combined.round(4)

## 1. Education as Moderator

Education is coded 1-5 (less than high school to postgraduate). We use the mean-centered version so that the base party promotion coefficients represent effects at the average education level.

In [ ]:
res_edu = fit_hte_models('education_c')

for outcome in OUTCOMES:
    r = res_edu[outcome]
    print(f'\n=== {OUTCOME_LABELS[outcome]} ===')
    print(f'N = {int(r.nobs)}, R² = {r.rsquared:.4f}')

print('\n--- Triple-interaction terms ---')
display(display_triple_terms(res_edu, 'education_c'))

## 2. Age as Moderator

In [ ]:
res_age = fit_hte_models('age_c')

for outcome in OUTCOMES:
    r = res_age[outcome]
    print(f'\n=== {OUTCOME_LABELS[outcome]} ===')
    print(f'N = {int(r.nobs)}, R² = {r.rsquared:.4f}')

print('\n--- Triple-interaction terms ---')
display(display_triple_terms(res_age, 'age_c'))

## 3. Political Interest as Moderator

In [ ]:
res_polint = fit_hte_models('political_interest_c')

for outcome in OUTCOMES:
    r = res_polint[outcome]
    print(f'\n=== {OUTCOME_LABELS[outcome]} ===')
    print(f'N = {int(r.nobs)}, R² = {r.rsquared:.4f}')

print('\n--- Triple-interaction terms ---')
display(display_triple_terms(res_polint, 'political_interest_c'))

## 4. Gender as Moderator

Gender is binary (0 = male, 1 = female). We fit the interaction model and also compare split-sample estimates side-by-side.

In [ ]:
res_gender = fit_hte_models('gender')

for outcome in OUTCOMES:
    r = res_gender[outcome]
    print(f'\n=== {OUTCOME_LABELS[outcome]} ===')
    print(f'N = {int(r.nobs)}, R² = {r.rsquared:.4f}')

print('\n--- Triple-interaction terms ---')
display(display_triple_terms(res_gender, 'gender'))

In [ ]:
# Split-sample comparison: men vs women
BASE_FORMULA = '{outcome} ~ 0 + C(party_id) + party_id_dem:headline_pro_dem + party_id_rep:headline_pro_rep'

coef_vars = {
    'party_id_dem:headline_pro_dem': 'Dem x Pro-Dem Headline',
    'party_id_rep:headline_pro_rep': 'Rep x Pro-Rep Headline',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, (gender_val, gender_label) in zip(axes, [(0, 'Male'), (1, 'Female')]):
    for i, outcome in enumerate(OUTCOMES):
        formula = BASE_FORMULA.format(outcome=outcome)
        res = fit_ols_clustered(formula, df, cluster_var='id', weight_var='weight',
                                subset=f'gender == {gender_val}')
        td = tidy(res)
        td = td[td['term'].isin(coef_vars.keys())].copy()
        td['label'] = td['term'].map(coef_vars)
        
        color = sns.color_palette('tab10')[i]
        y_pos = np.arange(len(td)) + i * 0.25 - 0.25
        
        ax.errorbar(td['estimate'], y_pos,
                    xerr=[td['estimate'] - td['conf_low'], td['conf_high'] - td['estimate']],
                    fmt='o', color=color, capsize=3, markersize=4,
                    label=OUTCOME_LABELS[outcome])
    
    ax.axvline(0, color='grey', linewidth=0.5)
    ax.set_yticks(range(len(coef_vars)))
    ax.set_yticklabels(list(coef_vars.values()))
    ax.set_title(f'{gender_label} Respondents')
    ax.set_xlabel('Coefficient Estimate')
    ax.legend(fontsize=7, loc='lower right')
    ax.invert_yaxis()

fig.suptitle('Split-Sample: Party Promotion by Gender', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 5. Summary Heatmap

All triple-interaction coefficients across moderators, outcomes, and parties. Color encodes sign and magnitude; asterisks denote statistical significance.

In [ ]:
# Collect all triple-interaction coefficients
all_mods = {
    'Education': ('education_c', res_edu),
    'Age': ('age_c', res_age),
    'Pol. Interest': ('political_interest_c', res_polint),
    'Gender': ('gender', res_gender),
}

heatmap_rows = []
for mod_label, (mod_var, res_dict) in all_mods.items():
    for outcome in OUTCOMES:
        td = tidy(res_dict[outcome])
        for party_term, party_label in [('party_id_dem:headline_pro_dem', 'Dem'),
                                         ('party_id_rep:headline_pro_rep', 'Rep')]:
            triple = f'{party_term}:{mod_var}'
            row = td[td['term'] == triple]
            if not row.empty:
                r = row.iloc[0]
                stars = '***' if r['p_value'] < 0.001 else '**' if r['p_value'] < 0.01 else '*' if r['p_value'] < 0.05 else ''
                heatmap_rows.append({
                    'Moderator': mod_label,
                    'Outcome': OUTCOME_LABELS[outcome],
                    'Party': party_label,
                    'coef': r['estimate'],
                    'label': f"{r['estimate']:.3f}{stars}",
                    'p_value': r['p_value'],
                })

heat_df = pd.DataFrame(heatmap_rows)
heat_df['row'] = heat_df['Moderator'] + ' (' + heat_df['Party'] + ')'

# Pivot for heatmap
pivot_coef = heat_df.pivot(index='row', columns='Outcome', values='coef')
pivot_label = heat_df.pivot(index='row', columns='Outcome', values='label')

# Desired row order
row_order = []
for mod in ['Education', 'Age', 'Pol. Interest', 'Gender']:
    for p in ['Dem', 'Rep']:
        row_order.append(f'{mod} ({p})')
pivot_coef = pivot_coef.reindex(row_order)
pivot_label = pivot_label.reindex(row_order)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_coef, annot=pivot_label, fmt='', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Coefficient'})
ax.set_title('Triple-Interaction Coefficients: Party Promotion x Moderator')
ax.set_ylabel('')
fig.tight_layout()
plt.show()

## 6. Marginal Effect Plots

For the continuous moderators (education and age), we plot how the party promotion effect varies across moderator values, with 95% confidence bands.

In [ ]:
# Education: marginal effect of party promotion across education levels
edu_vals = np.linspace(df['education_c'].min(), df['education_c'].max(), 50)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, outcome in zip(axes, OUTCOMES):
    res = res_edu[outcome]
    
    for base_term, triple_suffix, color, label in [
        ('party_id_dem:headline_pro_dem', 'party_id_dem:headline_pro_dem:education_c',
         '#377EB8', 'Democrat promotion'),
        ('party_id_rep:headline_pro_rep', 'party_id_rep:headline_pro_rep:education_c',
         '#E41A1C', 'Republican promotion'),
    ]:
        me = marginal_effect_at_values(res, base_term, triple_suffix, edu_vals)
        ax.plot(me['mod_value'], me['effect'], color=color, label=label)
        ax.fill_between(me['mod_value'], me['conf_low'], me['conf_high'],
                        alpha=0.2, color=color)
    
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Education (mean-centered)')
    ax.set_title(OUTCOME_LABELS[outcome])
    ax.legend(fontsize=8)

axes[0].set_ylabel('Party Promotion Effect')
fig.suptitle('Marginal Effect of Party Promotion by Education Level', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Age: marginal effect of party promotion across age
age_vals = np.linspace(df['age_c'].min(), df['age_c'].max(), 50)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, outcome in zip(axes, OUTCOMES):
    res = res_age[outcome]
    
    for base_term, triple_suffix, color, label in [
        ('party_id_dem:headline_pro_dem', 'party_id_dem:headline_pro_dem:age_c',
         '#377EB8', 'Democrat promotion'),
        ('party_id_rep:headline_pro_rep', 'party_id_rep:headline_pro_rep:age_c',
         '#E41A1C', 'Republican promotion'),
    ]:
        me = marginal_effect_at_values(res, base_term, triple_suffix, age_vals)
        ax.plot(me['mod_value'], me['effect'], color=color, label=label)
        ax.fill_between(me['mod_value'], me['conf_low'], me['conf_high'],
                        alpha=0.2, color=color)
    
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Age (mean-centered)')
    ax.set_title(OUTCOME_LABELS[outcome])
    ax.legend(fontsize=8)

axes[0].set_ylabel('Party Promotion Effect')
fig.suptitle('Marginal Effect of Party Promotion by Age', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 7. Interpretation

### Key findings:

1. **Education**: The triple-interaction terms test whether more-educated partisans show stronger or weaker party promotion. If the coefficient on `Dem x ProDem x Education` is positive, more-educated Democrats show *less* willingness to remove co-party headlines (stronger promotion). The magnitude and significance of these effects tells us whether education amplifies or dampens partisan bias in content moderation.

2. **Age**: Older respondents may have more crystallized partisan identities. The marginal effect plots show whether party promotion strengthens or weakens with age.

3. **Political Interest**: More politically interested respondents might be more motivated reasoners, potentially showing stronger party promotion. The triple-interaction captures whether heightened political engagement amplifies partisan bias.

4. **Gender**: The split-sample comparison visualizes whether men and women differ in the magnitude of their party promotion effects. Gender differences might reflect differential exposure to or engagement with political content online.

### Caveats
- The sample size limits statistical power for detecting interaction effects, so null results should not be interpreted as evidence of no heterogeneity.
- The moderators are not randomly assigned, so these associations are descriptive rather than causal.
- Multiple testing across 4 moderators x 3 outcomes x 2 parties = 24 tests increases the risk of false positives. The summary heatmap helps visualize the overall pattern rather than relying on individual p-values.